# 04 - OCR Vergleich: Docling EasyOCR vs. Docling VLM vs. externes VLM

Dieses Notebook vergleicht drei OCR-/Extraktionsansätze auf einem schwer lesbaren PDF mit Tabellen und Formeln:
- `notebooks/raw_data/lstm.pdf`

Ziele:
1. OCR mit **Docling + EasyOCR**
2. OCR/Extraktion mit **Docling VLM-Pipeline**
3. OCR mit **externem VLM** über LiteLLM

Am Ende vergleichen wir die Outputs qualitativ und mit einfachen Kennzahlen.


## 1) Konfiguration


In [18]:
from pathlib import Path
import os
import json
import re
import platform

def auto_use_mlx() -> bool:
    if platform.system() != "Darwin" or platform.machine() != "arm64":
        return False
    try:
        import mlx  # noqa: F401
        return True
    except Exception:
        return False

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
NOTEBOOKS_DIR = REPO_ROOT / 'notebooks'
RAW_DIR = NOTEBOOKS_DIR / 'raw_data'
OUT_DIR = NOTEBOOKS_DIR / 'processed' / 'ocr_compare'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATH = RAW_DIR / 'lstm_tables.pdf'

# Lade .env (API-Key / API-Base)
env_path = NOTEBOOKS_DIR / '.env'
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        os.environ.setdefault(k.strip(), v.strip())

API_BASE_URL = os.getenv('OPENAI_API_BASE', 'https://api.aisc.hpi.de/')
OPENAI_API_KEY_SET = bool(os.getenv('OPENAI_API_KEY'))

# Modellnamen ggf. an euer LiteLLM Deployment anpassen
EXTERNAL_VLM_MODEL = os.getenv('EXTERNAL_VLM_MODEL', 'openai/qwen3-vl-32b')
DOCLING_API_VLM_MODEL = os.getenv('DOCLING_API_VLM_MODEL', "qwen3-vl-32b")
DOCLING_API_VLM_URL = os.getenv('DOCLING_API_VLM_URL', API_BASE_URL.rstrip('/') + '/v1/chat/completions')

# Docling VLM Konfiguration
VLM_PRESET = os.getenv('DOCLING_VLM_PRESET', 'granite_docling')
VLM_USE_MLX = auto_use_mlx()

print('PDF:', PDF_PATH)
print('PDF exists:', PDF_PATH.exists())
print('OUT_DIR:', OUT_DIR)
print('API base:', API_BASE_URL)
print('OPENAI_API_KEY set:', OPENAI_API_KEY_SET)
print('EXTERNAL_VLM_MODEL:', EXTERNAL_VLM_MODEL)
print('DOCLING_API_VLM_MODEL:', DOCLING_API_VLM_MODEL)
print('DOCLING_API_VLM_URL:', DOCLING_API_VLM_URL)
print('DOCLING VLM preset:', VLM_PRESET)
print('DOCLING VLM use MLX:', VLM_USE_MLX)


PDF: /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/raw_data/lstm_tables.pdf
PDF exists: True
OUT_DIR: /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/processed/ocr_compare
API base: https://api.aisc.hpi.de/
OPENAI_API_KEY set: True
EXTERNAL_VLM_MODEL: openai/qwen3-vl-32b
DOCLING_API_VLM_MODEL: qwen3-vl-32b
DOCLING_API_VLM_URL: https://api.aisc.hpi.de/v1/chat/completions
DOCLING VLM preset: granite_docling
DOCLING VLM use MLX: True


In [2]:
if not PDF_PATH.exists():
    raise FileNotFoundError(f'PDF not found: {PDF_PATH}')
print('PDF found. Ready for OCR comparison.')


PDF found. Ready for OCR comparison.


## 2) Hilfsfunktionen

Wir vereinheitlichen Textbereinigung, Speichern und einfache Qualitätsmetriken.


In [3]:
def normalize_text(text: str) -> str:
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def save_text(name: str, text: str) -> Path:
    p = OUT_DIR / f'{name}.md'
    p.write_text(normalize_text(text), encoding='utf-8')
    return p

def basic_metrics(text: str) -> dict:
    t = normalize_text(text)
    return {
        'chars': len(t),
        'words': len(t.split()),
        'lines': len(t.splitlines()),
        'math_markers': sum(t.count(x) for x in ['$', '\\(', '\\)', '\\[', '\\]']),
        'table_markers': sum(t.count(x) for x in ['|', '\t']),
    }


## 3) OCR A: Docling OCR (RapidOCR / optional OcrMacOptions)

Stärken: konsistent im Docling-Workflow und flexibel je nach Workshop-Hardware.
Empfehlung: `RapidOcrOptions` als Standard, `OcrMacOptions` optional für macOS (oft schneller).


In [4]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions, OcrMacOptions

def ocr_with_docling(pdf_path: Path, engine: str = 'rapidocr', start_page: int = 1) -> tuple[str, dict]:
    opts = PdfPipelineOptions()
    opts.do_ocr = True
    if engine == 'mac':
        opts.ocr_options = OcrMacOptions(lang=["en-US"], force_full_page_ocr=True)
    elif engine == 'rapidocr':
        opts.ocr_options = RapidOcrOptions(lang=["english"], force_full_page_ocr=True)
    else:
        raise ValueError("engine must be 'rapidocr' or 'mac'")

    opts.do_table_structure = True
    opts.do_formula_enrichment = True

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=opts)
        }
    )
    result = converter.convert(str(pdf_path))
    doc = result.document

    md = doc.export_to_markdown()
    doc_json = doc.export_to_dict()

    return md, doc_json

# Optionen: 'rapidocr' | 'mac'
OCR_ENGINE = 'rapidocr'

docling_ocr_text, docling_ocr_json = ocr_with_docling(PDF_PATH, engine=OCR_ENGINE)
docling_ocr_out = save_text(f'04_s03_docling_{OCR_ENGINE}_ocr', docling_ocr_text)
(OUT_DIR / f'04_s03_docling_{OCR_ENGINE}_ocr.json').write_text(json.dumps(docling_ocr_json, ensure_ascii=False, indent=2), encoding='utf-8')
print('OCR engine:', OCR_ENGINE)
print('Saved:', docling_ocr_out)
print('Metrics:', basic_metrics(docling_ocr_text))


objc[24069]: Class AVFFrameReceiver is implemented in both /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/.venv/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x300fd03a8) and /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/.venv/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x3075e83a8). One of the two will be used. Which one is undefined.
objc[24069]: Class AVFAudioReceiver is implemented in both /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/.venv/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x300fd03f8) and /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/.venv/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x3075e83f8). One of the two will be used. Which one is undefined.
[INFO] 2026-02-19 14:16:26,088 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-19 14:16:26,125 [RapidOCR] download_file.py:60: File exists and is v

OCR engine: rapidocr
Saved: /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/processed/ocr_compare/04_s03_docling_rapidocr_ocr.md
Metrics: {'chars': 20207, 'words': 3377, 'lines': 130, 'math_markers': 24, 'table_markers': 150}


## 4) OCR B: Docling VLM (Granite Preset)

Hier nutzen wir das Preset `granite_docling` für VLM-basierte Extraktion.
Optional kann auf Apple Silicon per Flag ein MLX-Engine-Override aktiviert werden.
Wenn lokal kein passendes VLM-Setup verfügbar ist, nutze den externen VLM-Schritt als Fallback.


In [5]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import VlmConvertOptions, VlmPipelineOptions
from docling.pipeline.vlm_pipeline import VlmPipeline

def ocr_with_docling_vlm(pdf_path: Path, preset: str = 'granite_docling', use_mlx: bool = False) -> tuple[str, dict]:
    # Preset-basiert: empfohlen für reproduzierbare Workshop-Setups
    if use_mlx:
        from docling.datamodel.vlm_engine_options import MlxVlmEngineOptions
        vlm_options = VlmConvertOptions.from_preset(preset, engine_options=MlxVlmEngineOptions())
    else:
        vlm_options = VlmConvertOptions.from_preset(preset)

    vlm_pipe_opts = VlmPipelineOptions(vlm_options=vlm_options)
    vlm_pipe_opts.force_backend_text = False
    vlm_pipe_opts.images_scale = 1.0

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_cls=VlmPipeline,
                pipeline_options=vlm_pipe_opts,
            )
        }
    )
    result = converter.convert(str(pdf_path))
    doc = result.document
    md = doc.export_to_markdown()
    doc_json = doc.export_to_dict()

    return md, doc_json

docling_vlm_text = None
docling_vlm_json = None
try:
    docling_vlm_text, docling_vlm_json = ocr_with_docling_vlm(PDF_PATH, preset=VLM_PRESET, use_mlx=VLM_USE_MLX)
    docling_vlm_out = save_text('04_s04_docling_vlm_granite_ocr', docling_vlm_text)
    (OUT_DIR / '04_s04_docling_vlm_granite_ocr.json').write_text(json.dumps(docling_vlm_json, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Saved:', docling_vlm_out)
    print('Metrics:', basic_metrics(docling_vlm_text))
except Exception as e:
    print('Docling VLM OCR failed on this machine/environment.')
    print('Error:', e)


mx.metal.device_info is deprecated and will be removed in a future version. Use mx.device_info instead.


Saved: /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/processed/ocr_compare/04_s04_docling_vlm_granite_ocr.md
Metrics: {'chars': 17357, 'words': 3020, 'lines': 120, 'math_markers': 66, 'table_markers': 149}


## 5) OCR C: Externes VLM (LiteLLM) auf Seitenbildern

Hier vergleichen wir zwei externe Varianten:
1. Direkter LiteLLM-Aufruf pro Seite (Image->Text)
2. Docling `ApiVlmOptions` mit LiteLLM-Endpoint und explizitem `model` in `params`

Workshop-Default: `qwen3-vl-32b` am LiteLLM Endpoint.


In [6]:
from pdf2image import convert_from_path
import base64
from io import BytesIO
from litellm import completion

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    VlmPipelineOptions,
    PictureDescriptionApiOptions,
)
from docling.datamodel.pipeline_options_vlm_model import ApiVlmOptions, ResponseFormat
from docling.pipeline.vlm_pipeline import VlmPipeline

def pil_to_base64_data_url(img) -> str:
    buf = BytesIO()
    img.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    return f'data:image/png;base64,{b64}'

def ocr_with_external_vlm(pdf_path: Path, model: str) -> str:
    if not os.getenv('OPENAI_API_KEY'):
        raise ValueError('OPENAI_API_KEY fehlt für externes VLM.')

    images = convert_from_path(str(pdf_path))
    print(f'Extracted {len(images)} page images for external VLM OCR.')

    outputs = []
    for i, img in enumerate(images, start=1):
        img_url = pil_to_base64_data_url(img)
        prompt = (
            'Extract the page text as faithfully as possible. '
            'Keep LaTeX/math and table structure if visible. '
            'Return plain markdown text only.'
        )

        resp = completion(
            model=model,
            api_base=API_BASE_URL,
            api_key=os.getenv('OPENAI_API_KEY'),
            temperature=0.0,
            messages=[
                {
                    'role': 'user',
                    'content': [
                        {'type': 'text', 'text': prompt},
                        {'type': 'image_url', 'image_url': {'url': img_url}},
                    ],
                }
            ],
        )

        page_text = resp.choices[0].message.content
        outputs.append(f'## Page {i}\n\n{page_text.strip()}')

    return '\n\n'.join(outputs)

def ocr_with_docling_api_vlm(pdf_path: Path, model: str, annotate_pictures: bool = False) -> tuple[str, dict]:
    if not os.getenv('OPENAI_API_KEY'):
        raise ValueError('OPENAI_API_KEY fehlt für Docling ApiVlmOptions.')
    auth_headers = {"Authorization": f"Bearer {os.getenv('OPENAI_API_KEY')}"}
    api_opts = ApiVlmOptions(
        url=DOCLING_API_VLM_URL,
        headers=auth_headers,
        prompt='Extract page text faithfully. Preserve equations and tables. Return markdown.',
        response_format=ResponseFormat.MARKDOWN,
        params={
            'model': model,
            'temperature': 0.0,
        },
        timeout=120,
        concurrency=2,
    )

    pipe_opts = VlmPipelineOptions(
        enable_remote_services=True,
        vlm_options=api_opts,
    )
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_cls=VlmPipeline,
                pipeline_options=pipe_opts,
            )
        }
    )

    result = converter.convert(str(pdf_path))
    doc = result.document
    md = doc.export_to_markdown()
    doc_json = doc.export_to_dict()
    return md, doc_json




In [7]:
# # 1) direkter LiteLLM-Aufruf
external_vlm_text = ocr_with_external_vlm(PDF_PATH, model=EXTERNAL_VLM_MODEL)
external_vlm_out = save_text('04_s05_external_vlm_direct_ocr', external_vlm_text)
print('Saved:', external_vlm_out)


Extracted 6 page images for external VLM OCR.
Saved: /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/processed/ocr_compare/04_s05_external_vlm_direct_ocr.md


In [8]:
# 2) Docling ApiVlmOptions (inkl. JSON-Output)
docling_api_vlm_text, docling_api_vlm_json = ocr_with_docling_api_vlm(PDF_PATH, model=DOCLING_API_VLM_MODEL)
docling_api_vlm_out = save_text('04_s05_docling_api_vlm_ocr', docling_api_vlm_text)
(OUT_DIR / '04_s05_docling_api_vlm_ocr.json').write_text(json.dumps(docling_api_vlm_json, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', docling_api_vlm_out)


Saved: /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/processed/ocr_compare/04_s05_docling_api_vlm_ocr.md


## 6) Bildannotation separat + Merge in OCR-Kopie

Hier nutzen wir **nicht** die VLM-Picture-Description im VlmPipeline, sondern den robusteren Weg:
1. Bilder mit Standard-PDF-Pipeline extrahieren (`generate_picture_images=True`)
2. Jedes Bild extern mit `qwen3-vl-32b` beschreiben
3. Beschreibungen nach Reihenfolge in eine Kopie des OCR-Markdowns einfügen

Namenskonvention: `04_sXX_<pipeline>_<artifact>.<ext>` für schnelle Zuordnung im Workshop.


In [9]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling_core.types.doc import PictureItem

def extract_docling_pictures(pdf_path: Path):
    opts = PdfPipelineOptions()
    opts.do_ocr = False
    opts.generate_picture_images = True
    opts.images_scale = 1.0

    converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
    )

    result = converter.convert(str(pdf_path))
    doc = result.document

    pictures = []
    idx = 0
    for item, _level in doc.iterate_items():
        if isinstance(item, PictureItem) and item.image is not None and item.image.pil_image is not None:
            idx += 1
            pages = sorted({prov.page_no for prov in (item.prov or []) if hasattr(prov, 'page_no')})
            pictures.append({
                'picture_index': idx,
                'page_numbers': pages,
                'pil_image': item.image.pil_image,
            })
    return pictures

def describe_picture_with_external_vlm(pil_img, model: str) -> str:
    if not os.getenv('OPENAI_API_KEY'):
        raise ValueError('OPENAI_API_KEY fehlt für Bildannotation.')

    prompt = (
        'Describe this figure. '
        'If chart/diagram, explain structure and key takeaway. '
        'If formula image, transcribe math where possible. '
        'Return concise markdown.'
    )

    resp = completion(
        model=EXTERNAL_VLM_MODEL,
        api_base=API_BASE_URL,
        api_key=os.getenv('OPENAI_API_KEY'),
        temperature=0.0,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'text', 'text': prompt},
                {'type': 'image_url', 'image_url': {'url': pil_to_base64_data_url(pil_img)}},
            ],
        }],
    )
    return (resp.choices[0].message.content or '').strip()

def merge_image_descriptions_into_markdown(base_markdown: str, descriptions: list[dict]) -> str:
    parts = base_markdown.split('<!-- image -->')
    if len(parts) <= 1:
        return base_markdown

    merged = [parts[0]]
    for i in range(1, len(parts)):
        desc = descriptions[i - 1]['description'] if i - 1 < len(descriptions) else None
        page_txt = descriptions[i - 1].get('page_numbers', []) if i - 1 < len(descriptions) else []
        page_str = ', '.join(map(str, page_txt)) if page_txt else '-'

        if desc:
            merged.append(f'<!-- image -->\n\n**VLM Bildbeschreibung (Seiten: {page_str})**\n\n{desc}\n')
        else:
            merged.append('<!-- image -->\n')
        merged.append(parts[i])

    return ''.join(merged)


In [14]:

# Basis für Merge: OCR-Output aus Schritt 5
base_md_for_merge = docling_api_vlm_text

pictures = extract_docling_pictures(PDF_PATH)
print(f'Extracted picture items: {len(pictures)}')

picture_descriptions = []
for pic in pictures:
    desc = describe_picture_with_external_vlm(pic['pil_image'], model=EXTERNAL_VLM_MODEL)
    picture_descriptions.append({
        'picture_index': pic['picture_index'],
        'page_numbers': pic['page_numbers'],
        'description': desc,
    })

merged_md = merge_image_descriptions_into_markdown(base_md_for_merge, picture_descriptions)

merge_out = OUT_DIR / f'04_s06_docling_api_with_vlm_image_desc.md'
desc_out = OUT_DIR / '04_s06_external_vlm_image_descriptions.json'
merge_out.write_text(merged_md, encoding='utf-8')
desc_out.write_text(json.dumps(picture_descriptions, ensure_ascii=False, indent=2), encoding='utf-8')

print('Saved merged markdown:', merge_out)
print('Saved picture descriptions:', desc_out)
if picture_descriptions:
    print('\nFirst description preview:\n')
    for picture in picture_descriptions:
        print(f"Picture {picture['picture_index']} (pages: {', '.join(map(str, picture['page_numbers']))}):")
        print(picture['description'])
        print('---')


Extracted picture items: 2
Saved merged markdown: /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/processed/ocr_compare/04_s06_docling_api_with_vlm_image_desc.md
Saved picture descriptions: /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/processed/ocr_compare/04_s06_external_vlm_image_descriptions.json

First description preview:

Picture 1 (pages: 2):
```markdown
# Neural Network Unit Diagram

## Structure

This diagram illustrates a single neuron or processing unit in a neural network, showing the flow of inputs, computation, and outputs.

- **Inputs**: 
  - `net_c_i` (net input to unit `c_i`)
  - `w_c_i` (weights for input connections)
  - `w_out_i` (weights for output connections)

- **Internal Computation**:
  - `g`: activation function (sigmoid or similar)
  - `g y^in`: output of `g` applied to input `y^in`
  - `1.0`: constant bias term
  - `h`: another activation function (possibly sigmoid or linear)
  - `h y^out`: output of `h` applied to `y^out`

- 

In [15]:
enriched_json = dict(docling_api_vlm_json)  # copy

# 1) keep all annotations in one clear place
enriched_json["external_vlm_picture_descriptions"] = picture_descriptions

# 2) optional: attach by picture index into picture items (if present)
pics = enriched_json.get("pictures", [])
if isinstance(pics, list):
    desc_by_idx = {d["picture_index"]: d for d in picture_descriptions}
    for i, pic in enumerate(pics, start=1):
        if i in desc_by_idx and isinstance(pic, dict):
            pic["vlm_description"] = desc_by_idx[i]["description"]
            pic["vlm_description_pages"] = desc_by_idx[i].get("page_numbers", [])

# save
json_out = OUT_DIR / f"04_s06_docling_api_with_vlm_image_desc.json"
json_out.write_text(json.dumps(enriched_json, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", json_out)


Saved: /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/processed/ocr_compare/04_s06_docling_api_with_vlm_image_desc.json


## 7) Vergleich

Wir vergleichen die Verfahren mit einfachen Kennzahlen und Textvorschau.


In [16]:
results = {}
results[f'docling_{OCR_ENGINE}'] = docling_ocr_text
if docling_vlm_text:
    results['docling_vlm'] = docling_vlm_text
results['external_vlm'] = external_vlm_text
if docling_api_vlm_text:
    results['docling_api_vlm'] = docling_api_vlm_text

comparison = {name: basic_metrics(text) for name, text in results.items()}
print(json.dumps(comparison, indent=2, ensure_ascii=False))


{
  "docling_rapidocr": {
    "chars": 20207,
    "words": 3377,
    "lines": 130,
    "math_markers": 24,
    "table_markers": 150
  },
  "docling_vlm": {
    "chars": 17357,
    "words": 3020,
    "lines": 120,
    "math_markers": 66,
    "table_markers": 149
  },
  "external_vlm": {
    "chars": 21923,
    "words": 3574,
    "lines": 155,
    "math_markers": 204,
    "table_markers": 140
  },
  "docling_api_vlm": {
    "chars": 22156,
    "words": 3578,
    "lines": 141,
    "math_markers": 212,
    "table_markers": 140
  }
}


In [17]:
for name, text in results.items():
    print('\n' + '='*30)
    print(name.upper())
    print('='*30)
    print(normalize_text(text)[:2000])



DOCLING_RAPIDOCR
## LONG SHORT-TERM MEMORY

NEURAL COMPUTATION 9(8):1735-1780,1997

Sepp Hochreiter Fakultät fir Informatik Technische Universität Minchen 80290 Minchen,Germany hochreit@informatik.tu-muenchen.de http://www7.informatik.tu-muenchen.de/^hochreit

## Abstract

Learning to store information over extended time intervals via recurrent backpropagation takes a very long time, mostly due to insufficient, decaying error back fow. We briefy review Hochreiter's 1991 analysis of this problem, then address it by introducing a novel, efficient, gradient-based method called "Long Short-Term Memory" (LSTM). Truncating the gradient where this does not do harm, LSTM can learn to bridge minimal time lags in excess of 1000 discrete time steps by enforcing constant error fow through"constant error carrousels"within special units. Multiplicative gate units learn to open and close access to the constant error flow.LSTM islocal in space and time;its computational complexity per time step and w

## 8) Fazit

Typischerweise wirst du sehen:
- Docling OCR (RapidOCR oder OcrMacOptions): stabile lokale OCR-Basis ohne tesserocr-Build-Aufwand
- Docling VLM (Granite Preset): bessere Strukturerhaltung, optional mit MLX-Beschleunigung auf Apple Silicon
- Externes VLM (direkt): flexible Modellwahl über LiteLLM
- Externes VLM via Docling ApiVlmOptions: gleicher Endpoint, aber zusätzlich strukturierter Docling-JSON-Output

Nächster sinnvoller Schritt: Den besten OCR-Output als Ingestion-Quelle in Qdrant übernehmen.
